# Proyecto: Uso de API del clima

En este notebook se creará un proyecto que permita obtener el clima de una ciudad dada usando la API de visualcrossing, para luego ocupar en nuestro proyecto final.

Importamos las librerias necesarias

In [1]:
import pandas as pd
import numpy as np
import Funciones as f
import io
import requests
import time


Antes de poder descargar el clima que necesitamos veamos nuestros datos del data set de eventos de envios.

In [3]:
df2=pd.read_csv("delivery_events.csv")

In [4]:
df2.head()


,event_id,load_id,trip_id,event_type,facility_id,scheduled_datetime,actual_datetime,detention_minutes,on_time_flag,location_city,location_state
0,EVT00000001,LOAD00000001,TRIP00000001,Pickup,FAC00034,2022-01-01 18:00:00.000000,2022-01-01 20:58:55.918185,0,False,Houston,TX
1,EVT00000002,LOAD00000001,TRIP00000001,Delivery,FAC00046,2022-01-02 23:10:55.918185,2022-01-02 21:30:22.142060,230,True,Detroit,MI
2,EVT00000003,LOAD00000002,TRIP00000002,Pickup,FAC00015,2022-01-01 18:00:00.000000,2022-01-01 17:37:26.608430,62,True,Kansas City,MO
3,EVT00000004,LOAD00000002,TRIP00000002,Delivery,FAC00050,2022-01-02 02:13:26.608430,2022-01-02 05:55:46.238257,67,False,Indianapolis,IN
4,EVT00000005,LOAD00000003,TRIP00000003,Pickup,FAC00001,2022-01-01 08:00:00.000000,2022-01-01 07:28:02.169634,83,True,Columbus,OH


In [5]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170820 entries, 0 to 170819
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   event_id            170820 non-null  object
 1   load_id             170820 non-null  object
 2   trip_id             170820 non-null  object
 3   event_type          170820 non-null  object
 4   facility_id         170820 non-null  object
 5   scheduled_datetime  170820 non-null  object
 6   actual_datetime     170820 non-null  object
 7   detention_minutes   170820 non-null  int64 
 8   on_time_flag        170820 non-null  bool  
 9   location_city       170820 non-null  object
 10  location_state      170820 non-null  object
dtypes: bool(1), int64(1), object(9)
memory usage: 13.2+ MB


In [6]:
df2["scheduled_datetime"].sort_values().unique()

array(['2022-01-01 07:00:00.000000', '2022-01-01 08:00:00.000000',
       '2022-01-01 09:00:00.000000', ..., '2025-01-02 15:58:37.260345',
       '2025-01-02 16:31:42.056974', '2025-01-02 19:21:42.629833'],
      shape=(99624,), dtype=object)

In [7]:
df2["scheduled_datetime"].value_counts()

scheduled_datetime
2022-02-03 07:00:00.000000    17
2024-06-22 17:00:00.000000    16
2023-09-19 08:00:00.000000    16
2023-01-21 14:00:00.000000    16
2024-10-18 12:00:00.000000    16
                              ..
2025-01-02 03:24:53.753801     1
2024-12-31 17:03:19.351504     1
2025-01-01 20:22:31.911033     1
2024-12-31 21:55:14.777642     1
2024-08-18 08:52:01.475155     1
Name: count, Length: 99624, dtype: int64

In [8]:
df2["location_city"].unique()

array(['Houston', 'Detroit', 'Kansas City', 'Indianapolis', 'Columbus',
       'Portland', 'Phoenix', 'Denver', 'Seattle', 'Chicago', 'Las Vegas',
       'Philadelphia', 'Miami', 'Memphis', 'Minneapolis', 'Atlanta',
       'Charlotte', 'New York', 'Los Angeles', 'Dallas'], dtype=object)

In [9]:
f.categorico(df2,"location_city")

location_city
Kansas City     14562
Philadelphia    12031
Portland        11773
Seattle         11768
Columbus        10593
Charlotte       10267
Houston         10263
Los Angeles      8948
Minneapolis      8737
Miami            8709
New York         7485
Denver           7368
Chicago          7350
Las Vegas        7344
Dallas           7279
Memphis          5908
Indianapolis     5810
Atlanta          5763
Phoenix          4456
Detroit          4406
Name: count, dtype: int64
['Houston' 'Detroit' 'Kansas City' 'Indianapolis' 'Columbus' 'Portland'
 'Phoenix' 'Denver' 'Seattle' 'Chicago' 'Las Vegas' 'Philadelphia' 'Miami'
 'Memphis' 'Minneapolis' 'Atlanta' 'Charlotte' 'New York' 'Los Angeles'
 'Dallas']
20


Aquí observamos que hay datos de 20 lugares diferentes y por las fechas desde 2022-01-01 hasta 2025-01-02

Comenzamos con la descarga de los datos de clima en la página web https://www.visualcrossing.com/

In [ ]:
#API_KEY = 'Q43NJ469KNZMCJN23RWVLY46B'
#API_KEY = 'RNV9AH9KF9QJQHC7UML77WWC3'
#API_KEY= 'F3ZLLHBL2FGAJCC5YWNY383LK'
#API_KEY= 'W2KCWPT8GENCUPNVM2JR2YV4V'
API_KEY= 'S5SF4KJWTK55ZMF2WRLCH5QRE'
#API_KEY= 'E2NG6T5TEUW42QQAQHNY8MN47'

def obtener_clima(ciudad, fecha_inicio, fecha_fin):
    print(f"Consultando {ciudad}...")
    url = f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/{ciudad}/{fecha_inicio}/{fecha_fin}"
    params = {
        'unitGroup': 'us',
        'include': 'days',
        'key': API_KEY,
        'contentType': 'csv'
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        clima = pd.read_csv(io.StringIO(response.text))
        return clima
    elif response.status_code == 429:
        print("¡Límite diario alcanzado!")
        return None
    else:
        print(f"Error {response.status_code}")
        return None


Comenzamos descargando Houston

In [ ]:
houston_clima = obtener_clima("Houston", "2022-01-01", "2024-08-31")


In [ ]:
houston_clima2 = obtener_clima("Houston", "2024-09-01", "2025-01-02")

In [ ]:
houston_clima3=obtener_clima("Houston", "2025-01-01", "2025-01-02")

Unamos en un mismo DataFrame los datos de la ciudad de Houston

In [ ]:
houston_final=pd.concat([houston_clima,houston_clima2])

In [ ]:
houston_final.info()

In [ ]:
houston_final.nunique()

In [10]:
columnas= {'name':'ciudad', 'datetime':'fecha', 'tempmax':'tempmáx', 'tempmin':'tempmín', 'temp':'temp', 'feelslikemax':'sensación térmica máxima', 'feelslikemin':'sensación térmica mínima',
        'feelslike':'sensación térmica', 'dew':'rocío', 'humidity':'humedad', 'precip':'precipitación', 'precipprob':'probabilidad de precipitación',
        'precipcover':'cobertura de precipitación', 'preciptype':'tipo de precipitación', 'snow':'nieve', 'snowdepth':'profundidad de nieve', 'windgust':'ráfaga de viento',
        'windspeed':'velocidad del viento', 'winddir':'dirección del viento', 'sealevelpressure':'presión del nivel del mar', 'cloudcover':'nubosidad', 'visibility':'visibilidad',
        'solarradiation':'radiación solar', 'solarenergy':'energía solar', 'uvindex':'índice UV', 'severerisk':'riesgo severo', 'sunrise':'amanecer',
        'sunset':'atardecer', 'moonphase':'fase lunar', 'conditions':'condiciones', 'description':'descripción', 'icon':'icono', 'stations':'estaciones'}

In [ ]:
houston_final = houston_final.rename(columns=columnas)

In [ ]:
f.categorico(houston_final,"fecha")

In [ ]:
houston_final.columns

In [ ]:
houston_final=pd.read_csv("houston.csv")

Houston listo el clima. Vamos ahora con Detroit 

In [ ]:
Detroit_clima=obtener_clima("Detroit", "2022-01-01", "2024-08-31")

In [ ]:
Detroit_clima2=obtener_clima("Detroit", "2024-09-01", "2025-01-02")

In [ ]:
Detroit_final=pd.concat([Detroit_clima,Detroit_clima2])

In [ ]:
Detroit_final.info()

Vamos a descargar el clima de Kansas City

In [ ]:
Kansas_clima=obtener_clima("Kansas City", "2022-01-01", "2024-08-31")

In [ ]:
Kansas_clima2=obtener_clima("Kansas City","2024-09-01","2024-09-26")

In [ ]:
Kansas_clima3=obtener_clima("Kansas City","2024-09-27","2025-01-02")

In [ ]:
Kansas_final=pd.concat([Kansas_clima,Kansas_clima2])

Vamos a descargar el clima de Portland

In [ ]:
Portland_clima=obtener_clima("Portland", "2022-01-01", "2024-09-26")

In [ ]:
Portland_clima2=obtener_clima("Portland","2024-09-27", "2025-01-02")

Vamos a descargar el clima de Miami

In [ ]:
Miami_clima=obtener_clima("Miami", "2022-01-01", "2024-09-26")

In [ ]:
Miami_clima2=obtener_clima("Miami", "2024-09-27", "2025-01-02")

In [ ]:
Miami_final= pd.concat([Miami_clima,Miami_clima2])

In [ ]:
Miami_final.info()

Vamos a descargar el clima de New York

In [ ]:
New_clima=obtener_clima("New York", "2022-01-01", "2024-09-26")

In [ ]:
New_clima2=obtener_clima("New York", "2024-09-27", "2025-01-02")

In [ ]:
New_final= pd.concat([New_clima,New_clima2])

In [ ]:
New_final.info()

Vamos a descargar el clima de Los Angeles

In [ ]:
Angeles_clima=obtener_clima("Los Angeles", "2022-01-01", "2024-09-26")

In [ ]:
Angeles_clima2=obtener_clima("Los Angeles", "2024-09-27", "2025-01-02")

In [ ]:
Angeles_final= pd.concat([Angeles_clima,Angeles_clima2])

In [ ]:
Angeles_final.info()

In [ ]:
union_final= pd.concat([Portland_clima,Kansas_final])

Creamos el dataframe con los datos del clima

In [ ]:
union_final.to_csv('union_final.csv', index=False)

Cargamos el dataframe con los datos del clima para ver sus columnas y estructura

In [11]:
union_final=pd.read_csv("union_final.csv")

In [12]:
union_final = union_final.rename(columns=columnas)

In [13]:
union_final.head()

,ciudad,fecha,tempmáx,tempmín,temp,sensación térmica máxima,sensación térmica mínima,sensación térmica,rocío,humedad,...,energía solar,índice UV,riesgo severo,amanecer,atardecer,fase lunar,condiciones,descripción,icono,estaciones
0,Portland,2022-01-01,33.1,24.3,28.4,27.2,16.1,20.5,16.7,61.1,...,1.9,1,NaN,2022-01-01T07:50:58,2022-01-01T16:37:54,0.97,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"72698524242,KVUO,72698024229,KPDX,KTTD,E8319,7..."
1,Portland,2022-01-02,39.5,29.8,33.7,32.9,18.9,24.4,23.4,65.8,...,1.2,1,NaN,2022-01-02T07:50:58,2022-01-02T16:38:50,0.00,"Snow, Rain, Overcast",Cloudy skies throughout the day with rain or s...,rain,"72698524242,D6016,KVUO,72698024229,KPDX,KTTD,E..."
2,Portland,2022-01-03,48.1,37.8,40.9,41.7,32.4,36.1,36.7,85.0,...,0.8,1,NaN,2022-01-03T07:50:55,2022-01-03T16:39:49,0.03,"Rain, Overcast",Cloudy skies throughout the day with a chance ...,rain,"72698524242,D6016,KVUO,KPDX,72698024229,KTTD,E..."
3,Portland,2022-01-04,45.4,38.5,41.2,42.2,33.7,37.4,37.6,87.1,...,1.0,1,NaN,2022-01-04T07:50:50,2022-01-04T16:40:50,0.07,"Rain, Overcast",Cloudy skies throughout the day with a chance ...,snow,"72698524242,D6016,KVUO,72698024229,KPDX,KTTD,E..."
4,Portland,2022-01-05,43.1,38.1,40.0,42.0,30.7,37.8,37.4,90.5,...,0.6,0,NaN,2022-01-05T07:50:42,2022-01-05T16:41:52,0.10,"Rain, Overcast",Cloudy skies throughout the day with a chance ...,rain,"72698524242,KVUO,72698024229,KPDX,KTTD,E8319,7..."


In [14]:
houston_final=pd.read_csv("houston.csv")

In [17]:
houston_final=houston_final.rename(columns={"nombre":"ciudad","fecha y hora":"fecha"})

In [18]:
houston_final.head()

,ciudad,fecha,tempmáx,tempmín,temp,sensación térmica máxima,sensación térmica mínima,sensación térmica,rocío,humedad,...,energía solar,índice UV,riesgo severo,amanecer,atardecer,fase lunar,condiciones,descripción,icono,estaciones
0,Houston,2024-01-01,64.2,50.2,59.5,64.2,50.2,59.5,52.8,79.2,...,4.3,3,NaN,2024-01-01T07:17:03,2024-01-01T17:33:09,0.69,"Rain, Overcast",Cloudy skies throughout the day with rain clea...,rain,"KHOU,72059400188,KIAH,KMCJ,72244012918,D0326"
1,Houston,2024-01-02,52.3,43.1,48.0,52.3,37.0,44.8,40.1,74.2,...,2.9,2,NaN,2024-01-02T07:17:17,2024-01-02T17:33:52,0.72,"Rain, Overcast",Cloudy skies throughout the day with late afte...,rain,"KHOU,72059400188,KMCJ,72244012918,D0326"
2,Houston,2024-01-03,57.3,44.8,49.8,57.3,38.5,46.8,44.2,82.0,...,4.8,6,NaN,2024-01-03T07:17:30,2024-01-03T17:34:36,0.75,"Rain, Partially cloudy",Partly cloudy throughout the day with rain cle...,rain,"KHOU,72059400188,KIAH,KMCJ,72244012918,7224301..."
3,Houston,2024-01-04,59.3,44.8,52.4,59.3,40.2,50.9,41.2,67.2,...,4.1,6,NaN,2024-01-04T07:17:41,2024-01-04T17:35:20,0.78,Partially cloudy,Becoming cloudy in the afternoon.,partly-cloudy-day,"KHOU,72059400188,KIAH,KMCJ,72244012918,D0326"
4,Houston,2024-01-05,61.1,48.6,55.2,61.1,42.8,55.0,48.5,80.1,...,3.7,6,NaN,2024-01-05T07:17:50,2024-01-05T17:36:05,0.81,"Rain, Partially cloudy",Partly cloudy throughout the day with rain cle...,rain,"KHOU,72059400188,KMCJ,72244012918,D0326"


Hacemos el nuevo dataframe del clima

In [ ]:
union=pd.concat([houston_clima3,Angeles_final,Detroit_final,Miami_final,New_final,Portland_clima2,Kansas_clima3])

In [ ]:
union = union.rename(columns=columnas)

Creamos el dataframe con los datos del clima

In [ ]:
union.to_csv('union_clima.csv', index=False)

Unimos todos los datos del clima para generar un único archivo final con las 7 ciudades de estudio

In [20]:
union=pd.read_csv("union_clima.csv")

In [21]:
Clima=pd.concat([union,houston_final,union_final])

In [22]:
Clima.head()

,ciudad,fecha,tempmáx,tempmín,temp,sensación térmica máxima,sensación térmica mínima,sensación térmica,rocío,humedad,...,energía solar,índice UV,riesgo severo,amanecer,atardecer,fase lunar,condiciones,descripción,icono,estaciones
0,Houston,2025-01-01,61.2,46.8,54.4,61.2,41.0,53.3,39.0,57.1,...,4.1,6,NaN,2025-01-01T07:17:14,2025-01-01T17:33:42,0.06,Partially cloudy,Clearing in the afternoon.,partly-cloudy-day,"KHOU,72059400188,E6288,KIAH,KMCJ,72244012918,D..."
1,Houston,2025-01-02,59.4,48.4,55.4,59.4,43.9,55.0,48.1,76.5,...,2.2,1,NaN,2025-01-02T07:17:27,2025-01-02T17:34:25,0.10,Overcast,Cloudy skies throughout the day.,cloudy,"KHOU,72059400188,KIAH,KMCJ,72244012918,D0326"
2,Los Angeles,2022-01-01,60.5,45.3,52.4,60.5,40.9,51.2,30.1,43.4,...,11.2,5,NaN,2022-01-01T06:59:33,2022-01-01T16:55:18,0.97,Clear,Clear conditions throughout the day.,clear-day,"AV411,72288593197,KVNY,BVRC1,KBUR,KSMO,7228862..."
3,Los Angeles,2022-01-02,63.9,41.6,52.0,63.9,38.2,51.2,23.2,33.9,...,11.7,5,NaN,2022-01-02T06:59:44,2022-01-02T16:56:04,0.00,Clear,Clear conditions throughout the day.,clear-day,"AV411,72288593197,KVNY,BVRC1,KBUR,KSMO,7228862..."
4,Los Angeles,2022-01-03,63.0,38.1,50.4,63.0,38.1,50.4,30.4,48.2,...,11.9,5,NaN,2022-01-03T06:59:52,2022-01-03T16:56:51,0.03,Clear,Clear conditions throughout the day.,clear-day,"AV411,72288593197,KVNY,BVRC1,KBUR,KSMO,7228862..."


Creando el dataframe final con todos los datos del clima y hacer el análisis

In [23]:
Clima.to_csv("Clima.csv", index=False)